In [2]:
import kagglehub
import pandas as pd
import numpy as np
!pip install category_encoders
from category_encoders.hashing import HashingEncoder
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import LinearSVC
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, accuracy_score
import warnings

warnings.filterwarnings("ignore")

# Download latest version
path = kagglehub.dataset_download("aliafzal9323/chicago-crime-dataset-2024-2026")
#path is /kaggle/input/chicago-crime-dataset-2024-2026/chicago crimes.csv
print("Path to dataset files:", path)

#params: ID Case Number, Date, Block, IUCR, Primary Type, Description, Location Description, Arrest, Domestic, Ward, Community Area,
#FBI Code,  X Coordinate, Y Coordinate,  Year, Updated On, Latitude ,Longitude, Location

df = pd.read_csv(path + "/chicago crimes.csv")
#df = df[(df['Year'] == 2024) | (df['Year'] == 2025) | (df['Year'] == 2026)] #we might want to create 2026 as a seperate CSV for use as a training set

df.to_csv("crime_2024-2026.csv")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.9/85.9 kB 4.2 MB/s eta 0:00:00
Using Colab cache for faster access to the 'chicago-crime-dataset-2024-2026' dataset.
Path to dataset files: /kaggle/input/chicago-crime-dataset-2024-2026


In [3]:
df = pd.read_csv("crime_2024-2026.csv")
print(df.shape)

# -----------------------------
# CLEANING
# -----------------------------
df = df.drop([
    'ID', 'Case Number', 'Updated On', 'Latitude', 'Longitude',
    'X Coordinate', 'Y Coordinate', 'Date', 'Location'
], axis=1)

boolConv = lambda x: x.astype(str).str.strip().str.upper().map({'TRUE': True, 'FALSE': False})
strConv = lambda x: x.astype(str).str.strip()

df['Arrest'] = boolConv(df['Arrest'])
df['Domestic'] = boolConv(df['Domestic'])

# Convert categorical text
df['Primary Type'] = strConv(df['Primary Type'])
df['Location Description'] = strConv(df['Location Description'])
df['Description'] = strConv(df['Description'])
df['Block'] = strConv(df['Block'])

# Convert numeric fields
for c in ['Beat', 'District', 'Ward', 'Community Area', 'FBI Code', 'Year']:
    df[c] = pd.to_numeric(df[c], errors='coerce')

# -----------------------------
# TARGET ENCODING (ARREST RATE)
# -----------------------------
df['PrimaryType_arrest_rate'] = df['Primary Type'].map(df.groupby('Primary Type')['Arrest'].mean())
df['LocationDesc_arrest_rate'] = df['Location Description'].map(df.groupby('Location Description')['Arrest'].mean())
df['IUCR_arrest_rate'] = df['IUCR'].map(df.groupby('IUCR')['Arrest'].mean())

# -----------------------------
# FREQUENCY ENCODING
# -----------------------------
df['PrimaryType_freq'] = df['Primary Type'].map(df['Primary Type'].value_counts())
df['LocationDesc_freq'] = df['Location Description'].map(df['Location Description'].value_counts())
df['IUCR_freq'] = df['IUCR'].map(df['IUCR'].value_counts())

# -----------------------------
# HASHING ENCODING (BIG FIELDS)
# -----------------------------
hash_encoder = HashingEncoder(n_components=16)
hashed = hash_encoder.fit_transform(df[['Block', 'Description']])
df = pd.concat([df, hashed], axis=1)

# Drop raw high-cardinality columns
df = df.drop(['Primary Type', 'Location Description', 'IUCR', 'Block', 'Description'], axis=1)

# -----------------------------
# TRAIN/TEST SPLIT
# -----------------------------
for i in range(12, 24, 3):
  train = df[(df['Year'] == 2002+i) | (df['Year'] == 2003+i)]
  test = df[df['Year'] == 2004+i]
  print(train.shape)
  print(test.shape)
  X_train = train.drop(['Domestic', 'Arrest'], axis=1)
  X_test  = test.drop(['Domestic', 'Arrest'], axis=1)

  print(X_train.shape)
  print(X_test.shape)

  y_train = train['Arrest']
  y_test  = test['Arrest']
  ###############

  # -----------------------------
  # STACKING MODEL (Decision Tree + Linear SVC with Logistic Regression)
  # -----------------------------

  estimators = [
      ('dt', DecisionTreeClassifier(class_weight='balanced')),
      ('lin_svc', LinearSVC(class_weight='balanced'))
  ]

  stack = StackingClassifier(
      estimators=estimators,
      final_estimator=LogisticRegression(max_iter=300, class_weight='balanced'),
      cv=5
  )

  model = Pipeline([
      ('imputer', SimpleImputer(strategy='median')),
      ('stack', stack)
  ])

  model.fit(X_train, y_train)

  y_pred = model.predict(X_test)
  m1 = accuracy_score(y_test, y_pred)
  print(classification_report(y_test, y_pred))
  print(m1)

  # -----------------------------
  # STACKING MODEL (XGBoost + Linear SVC with MLP Classifier)
  # -----------------------------

  estimators = [
      ('xgb', XGBClassifier(n_estimators=3, max_depth=2, learning_rate=1, objective='binary:logistic')),
      ('lin_svc', LinearSVC(class_weight='balanced'))
  ]

  stack = StackingClassifier(
      estimators=estimators,
      final_estimator=MLPClassifier(activation='relu', solver='adam', hidden_layer_sizes=(100, 100, 100)),
      cv=5
  )

  model = Pipeline([
      ('imputer', SimpleImputer(strategy='median')),
      ('stack', stack)
  ])

  model.fit(X_train, y_train)

  y_pred = model.predict(X_test)
  m2 = accuracy_score(y_test, y_pred)
  print(classification_report(y_test, y_pred))
  print(m2)

  # -----------------------------
  # STACKING MODEL (Decision Tree + Linear SVC with Logistic Regression) 2-CV
  # -----------------------------

  estimators = [
      ('dt', DecisionTreeClassifier(class_weight='balanced')),
      ('lin_svc', LinearSVC(class_weight='balanced'))
  ]

  stack = StackingClassifier(
      estimators=estimators,
      final_estimator=LogisticRegression(max_iter=300, class_weight='balanced'),
      cv=2
  )

  model = Pipeline([
      ('imputer', SimpleImputer(strategy='median')),
      ('stack', stack)
  ])

  model.fit(X_train, y_train)

  y_pred = model.predict(X_test)
  m3 = accuracy_score(y_test, y_pred)
  print(classification_report(y_test, y_pred))
  print(m3)

  # -----------------------------
  # STACKING MODEL (XGBoost + Linear SVC with MLP Classifier) 2-CV
  # -----------------------------

  estimators = [
      ('xgb', XGBClassifier(n_estimators=3, max_depth=2, learning_rate=1, objective='binary:logistic')),
      ('lin_svc', LinearSVC(class_weight='balanced'))
  ]

  stack = StackingClassifier(
      estimators=estimators,
      final_estimator=MLPClassifier(activation='relu', solver='adam', hidden_layer_sizes=(100, 100, 100)),
      cv=2
  )

  model = Pipeline([
      ('imputer', SimpleImputer(strategy='median')),
      ('stack', stack)
  ])

  model.fit(X_train, y_train)

  y_pred = model.predict(X_test)
  m4 = accuracy_score(y_test, y_pred)



  report_m4 = classification_report(y_test, y_pred)
  print("run: " + str(2002+i) +"-" + + str(2002+i))
  print("\n" + report_m4)
  print(m4)


print("done")
#plt.bar(range(4), [m1, m2, m3, m4])
#plt.show()

(8500901, 23)
(540767, 31)
(269957, 31)
(540767, 29)
(269957, 29)
              precision    recall  f1-score   support

       False       0.89      0.68      0.77    216879
        True       0.34      0.66      0.45     53078

    accuracy                           0.68    269957
   macro avg       0.62      0.67      0.61    269957
weighted avg       0.78      0.68      0.71    269957

0.6804305870935001
              precision    recall  f1-score   support

       False       0.88      0.99      0.93    216879
        True       0.92      0.46      0.62     53078

    accuracy                           0.89    269957
   macro avg       0.90      0.73      0.78    269957
weighted avg       0.89      0.89      0.87    269957

0.8869782965435236
              precision    recall  f1-score   support

       False       0.86      0.67      0.75    216879
        True       0.29      0.57      0.39     53078

    accuracy                           0.65    269957
   macro avg       0.58 

TypeError: bad operand type for unary +: 'str'